In [0]:
# ===========================================
# Notebook Name:
# 00_initialize_pipeline_run
#
# Purpose:
# First task of every Workflow run. Mints a
# pipeline_run_id, records a "running" row in
# pipeline_run_log, and exposes job
# parameters plus the run's start time to
# downstream tasks via taskValues.
#
# Output:
# pokemon.ops.pipeline_run_log
# ===========================================

In [0]:
from pyspark.sql import functions as F

PIPELINE_RUN_LOG_TABLE = (
    "pokemon.ops.pipeline_run_log"
)
PIPELINE_STATE_TABLE = (
    "pokemon.ops.pipeline_state"
)

dbutils.widgets.text("job_run_id", "")
dbutils.widgets.text(
    "pipeline_name",
    "daily_pokemon_lakehouse_pipeline",
)
dbutils.widgets.text(
    "min_event_date", "2026-01-01"
)
dbutils.widgets.text(
    "request_interval_seconds", "1.0"
)
dbutils.widgets.text("max_pages", "200")
dbutils.widgets.dropdown(
    "force_refetch",
    "false",
    ["true", "false"],
)
dbutils.widgets.dropdown(
    "rebuild_gold",
    "true",
    ["true", "false"],
)
dbutils.widgets.dropdown(
    "rebuild_similarity",
    "true",
    ["true", "false"],
)
dbutils.widgets.dropdown(
    "rebuild_archetypes",
    "true",
    ["true", "false"],
)

job_run_id = dbutils.widgets.get(
    "job_run_id"
).strip()
pipeline_name = dbutils.widgets.get(
    "pipeline_name"
).strip()

print("Output:", PIPELINE_RUN_LOG_TABLE)
print("job_run_id   :", job_run_id or "(none, interactive run)")
print("pipeline_name:", pipeline_name)

In [0]:
import uuid

pipeline_run_id = (
    job_run_id
    if job_run_id
    else str(uuid.uuid4())
)

print("pipeline_run_id:", pipeline_run_id)

In [0]:
if spark.catalog.tableExists(
    PIPELINE_STATE_TABLE
):
    previous_state_row = (
        spark.table(PIPELINE_STATE_TABLE)
        .filter(
            (F.col("pipeline_name") == pipeline_name)
            & (F.col("stage_name") == "overall")
        )
        .select(
            "last_run_status",
            "last_success_at",
            "consecutive_failure_count",
        )
        .first()
    )

else:
    previous_state_row = None

if previous_state_row is not None:
    print(
        "Previous run status     :",
        previous_state_row["last_run_status"],
    )
    print(
        "Previous success at     :",
        previous_state_row["last_success_at"],
    )
    print(
        "Consecutive failures    :",
        previous_state_row[
            "consecutive_failure_count"
        ],
    )
else:
    print(
        "No previous pipeline_state row for "
        f"pipeline_name={pipeline_name} "
        "(first run)."
    )

In [0]:
run_log_df = spark.createDataFrame(
    [{
        "pipeline_run_id": pipeline_run_id,
        "job_run_id": job_run_id or None,
        "task_name": "pipeline",
        "status": "running",
    }],
    schema="""
    pipeline_run_id STRING,
    job_run_id STRING,
    task_name STRING,
    status STRING
    """,
).withColumn(
    "started_at", F.current_timestamp()
)

run_log_df.createOrReplaceTempView(
    "staged_pipeline_run_log"
)

spark.sql(f"""
MERGE INTO {PIPELINE_RUN_LOG_TABLE} AS target
USING staged_pipeline_run_log AS source
ON  target.pipeline_run_id = source.pipeline_run_id
AND target.task_name = source.task_name

WHEN MATCHED THEN UPDATE SET
    target.job_run_id = source.job_run_id,
    target.status = source.status,
    target.started_at = source.started_at,
    target.finished_at = NULL

WHEN NOT MATCHED THEN INSERT (
    pipeline_run_id,
    job_run_id,
    task_name,
    status,
    started_at
)
VALUES (
    source.pipeline_run_id,
    source.job_run_id,
    source.task_name,
    source.status,
    source.started_at
)
""")

print(
    "pipeline_run_log: recorded 'running' "
    f"row for pipeline_run_id={pipeline_run_id}"
)

In [0]:
started_at_iso = (
    spark.table(PIPELINE_RUN_LOG_TABLE)
    .filter(
        (F.col("pipeline_run_id") == pipeline_run_id)
        & (F.col("task_name") == "pipeline")
    )
    .select("started_at")
    .first()["started_at"]
    .isoformat()
)

dbutils.jobs.taskValues.set(
    key="pipeline_run_id",
    value=pipeline_run_id,
)
dbutils.jobs.taskValues.set(
    key="pipeline_name",
    value=pipeline_name,
)
dbutils.jobs.taskValues.set(
    key="started_at_iso",
    value=started_at_iso,
)

print(
    "taskValues set: pipeline_run_id, "
    "pipeline_name, started_at_iso"
)